In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    os.getenv("DATASET_DB_URL", "mysql+pymysql://root:@127.0.0.1:3307/family_finance_mono?charset=utf8mb4")
)

In [2]:
# Gastos e ingresos
df = pd.read_sql("""
SELECT 
   `year_month`,
   BIN_TO_UUID(family_id) as family_id,
   SUM(total_income) AS total_income,
   SUM(total_expenses) AS total_expenses
   
FROM (
   SELECT 
      DATE_FORMAT(income_entries.date, '%%Y-%%m') AS `year_month`,
      family_members.family_id,
      SUM(income_entries.amount) AS total_income,
      0 AS total_expenses
   FROM income_entries
   LEFT JOIN family_members ON income_entries.family_member_id = family_members.id
   WHERE income_entries.is_active = 1
   GROUP BY `year_month`, family_members.family_id

   UNION ALL

   SELECT 
      DATE_FORMAT(expenses.date, '%%Y-%%m') AS `year_month`,
      family_members.family_id,
      0 AS total_income,
      SUM(expenses.amount) AS total_expenses
   FROM expenses
   LEFT JOIN family_members ON expenses.family_member_id = family_members.id
   GROUP BY `year_month`, family_members.family_id
) AS combined

GROUP BY `year_month`, family_id
ORDER BY `year_month`;
""", engine)

In [3]:
# Sacamos el balance
df['balance'] = df['total_income'] - df['total_expenses']

In [4]:
df.head(10)

,year_month,family_id,total_income,total_expenses,balance
0,2022-01,b006e9da-0024-426a-8f7d-dfa84d676ff7,0.00,346.81,-346.81
1,2022-01,d0000000-0000-0000-0000-000000000000,1261.72,2467.38,-1205.66
2,2022-01,d1111111-1111-1111-1111-111111111111,15006.92,7113.56,7893.36
3,2022-01,d2222222-2222-2222-2222-222222222222,2528.79,2603.01,-74.22
4,2022-01,d3333333-3333-3333-3333-333333333333,10514.43,4026.90,6487.53
5,2022-01,d4444444-4444-4444-4444-444444444444,3503.17,3107.21,395.96
6,2022-01,d5555555-5555-5555-5555-555555555555,3475.46,3491.63,-16.17
7,2022-01,d6666666-6666-6666-6666-666666666666,2428.74,2293.79,134.95
8,2022-01,d7777777-7777-7777-7777-777777777777,1175.55,3876.05,-2700.50
9,2022-01,d8888888-8888-8888-8888-888888888888,3288.71,14161.51,-10872.80


In [ ]:
# Extraemos los datos a un csv
df.to_csv('../../data/family-finance-family-data.csv', index=False)